# Statistical Analysis: Cochran's Q and McNemar's Tests

This notebook runs statistical significance tests to compare
LLaMA 3.3's performance across the four dataset conditions.
Cochran's Q Test is used for overall comparison, followed by
pairwise McNemar's Tests with Bonferroni correction.

In [ ]:

# STEP 1: Install required library

!pip install statsmodels

# STEP 2: Upload your Excel file
# Run this cell, click "Choose Files", and upload your .xlsx

from google.colab import files
uploaded = files.upload()


# STEP 3: Run the full analysis

import pandas as pd
import numpy as np
from statsmodels.stats.contingency_tables import cochrans_q
from statsmodels.stats.contingency_tables import mcnemar
from itertools import combinations

# Load data
df = pd.read_excel(list(uploaded.keys())[0])
print("Columns found:", df.columns.tolist())
print("Shape:", df.shape)
print("\nAccuracy per corpus:")
print(df.mean().round(4) * 100)

# Cochran's Q Test (overall)
print("\n" + "="*50)
print("COCHRAN'S Q TEST (Overall)")
print("="*50)
result = cochrans_q(df.values)
print(f"Q statistic : {result.statistic:.4f}")
print(f"p-value     : {result.pvalue:.4f}")
if result.pvalue < 0.05:
    print("Result: SIGNIFICANT ✓ (at least one corpus differs)")
else:
    print("Result: NOT significant")

# Pairwise McNemar's Tests (with Bonferroni correction)
print("\n" + "="*50)
print("PAIRWISE McNEMAR'S TESTS (Bonferroni corrected)")
print("="*50)

cols = df.columns.tolist()
pairs = list(combinations(cols, 2))
n_pairs = len(pairs)
alpha_corrected = 0.05 / n_pairs
print(f"Number of pairs: {n_pairs}")
print(f"Bonferroni-corrected alpha: {0.05}/{n_pairs} = {alpha_corrected:.4f}\n")

for col1, col2 in pairs:
    # Build 2x2 contingency table
    a = ((df[col1] == 1) & (df[col2] == 1)).sum()  # both correct
    b = ((df[col1] == 1) & (df[col2] == 0)).sum()  # col1 correct, col2 wrong
    c = ((df[col1] == 0) & (df[col2] == 1)).sum()  # col1 wrong, col2 correct
    d = ((df[col1] == 0) & (df[col2] == 0)).sum()  # both wrong

    table = [[a, b], [c, d]]
    result_mc = mcnemar(table, exact=True)

    sig = "✓ SIGNIFICANT" if result_mc.pvalue < alpha_corrected else "✗ not significant"
    print(f"{col1} vs {col2}")
    print(f"  p-value = {result_mc.pvalue:.4f}  →  {sig}")
    print()

print("="*50)
print("SUMMARY TABLE")
print("="*50)
print(f"\n{'Corpus':<25} {'Accuracy':>10}")
print("-"*35)
for col in cols:
    print(f"{col:<25} {df[col].mean()*100:>9.1f}%")

Saving Mcnemar_test_corrected.xlsx to Mcnemar_test_corrected.xlsx
Columns found: ['Monolingual English', 'Monolingual Sindhi', 'English Matrix', 'Sindhi Matrix']
Shape: (100, 4)

Accuracy per corpus:
Monolingual English    65.0
Monolingual Sindhi     60.0
English Matrix         63.0
Sindhi Matrix          65.0
dtype: float64

COCHRAN'S Q TEST (Overall)
Q statistic : 2.1613
p-value     : 0.5396
Result: NOT significant

PAIRWISE McNEMAR'S TESTS (Bonferroni corrected)
Number of pairs: 6
Bonferroni-corrected alpha: 0.05/6 = 0.0083

Monolingual English vs Monolingual Sindhi
  p-value = 0.3323  →  ✗ not significant

Monolingual English vs English Matrix
  p-value = 0.7905  →  ✗ not significant

Monolingual English vs Sindhi Matrix
  p-value = 1.0000  →  ✗ not significant

Monolingual Sindhi vs English Matrix
  p-value = 0.6072  →  ✗ not significant

Monolingual Sindhi vs Sindhi Matrix
  p-value = 0.2266  →  ✗ not significant

English Matrix vs Sindhi Matrix
  p-value = 0.8238  →  ✗ not signi